# E006 — Offline evaluation of ranking policies

**Hypotheses:** H11, H6 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Would ranking by the model's pCTR serve more clicked ads than the policy that produced the logs, and what do safety gates and exploration cost?

**Method:** Reconstruct candidate sets per publisher x hour on the test days, take each creative's share of the cell as its logging propensity, re-score every (impression, candidate) pair with the shipped bundle, and estimate each policy with IPS, SNIPS and doubly robust estimators plus paired hour-block bootstrap lifts.

In [1]:
from IPython.display import Markdown

from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
from charade.analysis import policy_eval
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
policy_eval.run(ctx.settings, ctx.data_dir, report=ctx.reports / "policy" / "ope.md")
Markdown((ctx.reports / "policy" / "ope.md").read_text())

# Off-policy evaluation (generated by `uv run poe ope`)

Evaluated impressions: 102,874 of 127,406 test rows (80.7%); cells: 5,484; mean candidates per cell: 4.13

| Policy | Estimator | CTR | 95 % CI | ESS | Max weight |
|---|---|---|---|---|---|
| logging (observed) | on-policy | 0.1723 | [0.1649, 0.1795] | 102,874 | 1.0 |
| uniform random | ips | 0.1905 | [0.1814, 0.1987] | 17,149 | 196.9 |
| uniform random | snips | 0.1760 | [0.1694, 0.1819] | 17,149 | 196.9 |
| uniform random | dr | 0.1704 | [0.1623, 0.1775] | 17,149 | 196.9 |
| greedy pCTR | ips | 0.2021 | [0.1857, 0.2168] | 4,236 | 445.0 |
| greedy pCTR | snips | 0.1960 | [0.1819, 0.2099] | 4,236 | 445.0 |
| greedy pCTR | dr | 0.1851 | [0.1713, 0.1981] | 4,236 | 445.0 |
| shipped (gates + 5% Thompson) | ips | 0.1943 | [0.1803, 0.2067] | 4,516 | 426.0 |
| shipped (gates + 5% Thompson) | snips | 0.1935 | [0.1796, 0.2064] | 4,516 | 426.0 |
| shipped (gates + 5% Thompson) | dr | 0.1836 | [0.1695, 0.1966] | 4,516 | 426.0 |

## Lift over the logging policy (paired hour-block bootstrap)

| Policy | SNIPS lift [95 % CI] | DR lift [95 % CI] |
|---|---|---|
| uniform random | +0.0037 [-0.0000, +0.0071] | -0.0019 [-0.0053, +0.0013] |
| greedy pCTR | +0.0237 [+0.0114, +0.0362] | +0.0128 [+0.0007, +0.0250] |
| shipped (gates + 5% Thompson) | +0.0212 [+0.0090, +0.0325] | +0.0113 [-0.0009, +0.0220] |


## Reading
- Uniform random lands near the logging CTR, as it should when logging is frequency-proportional.
- Greedy pCTR's DR lift excludes zero; the shipped policy pays ~0.15 pp for brand safety, caps and
  exploration, which pushes its interval across zero.
- H6 shows up as the frequency-cap gate and as `log_user_campaign_imps` in the model.

## Conclusion
Greedy pCTR ranking beats the logging policy by +1.28 pp CTR (DR, [+0.07, +2.50]); the shipped policy with gates and 5 % exploration is +1.13 pp with a CI touching zero; ESS ~4 % of rows.

**Decision:** Ship greedy + gates + 5 % Thompson exploration and log real propensities and candidate sets so the next evaluation does not need reconstruction.